# 1: Importando bibliotecas e fazendo a leitura do arquivo de dados

In [32]:
#Importando bibliotecas necessárias
import pandas as pd
import numpy as np
import warnings

# Configurações para exibição de dados
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', None)

# Carregando a base de dados
#df= pd.read_csv('dados_raw.csv') #Este método não deu certo porque o arquivo usa ponto e vírgula como separador
#df = pd.read_csv('dados_raw.csv', sep=';') #Este também não deu certo porque o arquivo não está no mesmo diretório do notebook
caminho = r'G:\Meu Drive\ScTec\Profissionalizar\Miniprojeto - Módulo 1\data\dados_raw.csv'
df = pd.read_csv(caminho, sep=';')

# Imprimindo mensagem de sucesso
print("Bibliotecas e base de dados importados com sucesso!")

Bibliotecas e base de dados importados com sucesso!


# 2: Tendo uma ideia geral de como é a base de dados

**Estas são as informações sobre as colunas, retiradas da documentação da base de dados:**

1. DATA: Data da compra;
2. CO_ID: Identificação do número de compra (número da nota fiscal);
3. CL_ID: Identificação do cliente (número do cliente);
4. CL_GENERO: Sexo biológico informado pelo cliente;
5. CL_EC: Estado civil do cliente:
    1: Casado ou união estával;
    2: Divorciado;
    3: Separado;
    4. Solteiro;
    5: Viúvo.
6. CL_FHL: Número de filhos do cliente;
7. CL_SEG: Segmentação econômica do cliente (classe A, B ou C);
8. PR_ID: Código do produto (SKU) adquirido;
9. PR_CAT: Categoria do produto adquirido;
10. PR_NOME: Nome do produto adquirido.


In [33]:
print("-" * 50)
print("Exibindo as primeiras linhas do DataFrame com head:")
print(df.head())

print("\n" + "-" * 100)
print("Exibindo os dados com describe:")
print(df.describe())

print("\n" + "-" * 100)
print("Exibindo os dados com info:")
print(df.info())

print("\n" + "-" * 100)
print(f"Tamanho: {df.shape[0]} linhas e {df.shape[1]} colunas")

#Problemas encontrados: 
# 4 últimas colunas são inúteis (provavelmente erro no download dos dados);
# Coluna Data em string;
# Coluna de estado civil apresenta números que representam categorias, o que pode dificultar a análise;
# Há poucas colunas com dados numéricos, então a estatística do info não é tão útil.
# O ponto positivo é que não há nulos, a princípio, em nenhuma das 10 colunas com dados úteis.


--------------------------------------------------
Exibindo as primeiras linhas do DataFrame com head:
         DATA  CO_ID  CL_ID CL_GENERO  CL_EC  CL_FHL CL_SEG  PR_ID     PR_CAT               PR_NOME  Unnamed: 10  Unnamed: 11  Unnamed: 12  Unnamed: 13
0  01/02/2019   1000    534         M      4       1      C     67    BEBIDAS  REFRIGERANTE GUARANA          NaN          NaN          NaN          NaN
1  01/02/2019   1000    534         M      4       1      C     70    BEBIDAS   REFRIGERANTE OUTROS          NaN          NaN          NaN          NaN
2  01/02/2019   1000    534         M      4       1      C    178    HIGIENE       LENCO UMEDECIDO          NaN          NaN          NaN          NaN
3  01/02/2019   1000    534         M      4       1      C      4  ALIMENTOS               ABACAXI          NaN          NaN          NaN          NaN
4  01/02/2019   1000    534         M      4       1      C    175    LIMPEZA     LIMPADOR MULTIUSO          NaN          NaN          Na

In [34]:
# A leitura das colunas está difícil, então vou renomeá-las para facilitar a manipulação dos dados
df = df.drop(columns=['Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13'], errors='ignore')
df.columns = ['data', 'id_compra','id_cliente','cl_genero','cl_estado_civil','num_filhos','cl_classe','id_produto','cat_produto','nome_produto']

df.head(5)

,data,id_compra,id_cliente,cl_genero,cl_estado_civil,num_filhos,cl_classe,id_produto,cat_produto,nome_produto
0,01/02/2019,1000,534,M,4,1,C,67,BEBIDAS,REFRIGERANTE GUARANA
1,01/02/2019,1000,534,M,4,1,C,70,BEBIDAS,REFRIGERANTE OUTROS
2,01/02/2019,1000,534,M,4,1,C,178,HIGIENE,LENCO UMEDECIDO
3,01/02/2019,1000,534,M,4,1,C,4,ALIMENTOS,ABACAXI
4,01/02/2019,1000,534,M,4,1,C,175,LIMPEZA,LIMPADOR MULTIUSO


# 3. Analisando se há nulos, duplicatas, e outros problemas

**Na parte anterior já havíamos visto alguns problemas:**
- Coluna data está como string;
- A coluna cl_estado_civil tem valores inteiros, mas que representam categorias, conforme visto na documentação;
- Os dados parecem estar todos formatados para ter tudo em maiúsculas, mas cabe verificar este ponto também.

In [35]:
print(f"Número de valores nulos por coluna: {df.isnull().sum()}") # Realmente não há valores nulos.

#Achei estranho não sair nenhum valor nulo, então vou puxar uma função:
import os
if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
from utils.utils import encontrar_falsos_nulos
encontrar_falsos_nulos(df) # O resultado não mostrou nenhum falso nulo.

Número de valores nulos por coluna: data               0
id_compra          0
id_cliente         0
cl_genero          0
cl_estado_civil    0
num_filhos         0
cl_classe          0
id_produto         0
cat_produto        0
nome_produto       0
dtype: int64
----------------------------------------------------------------------------------------------------
   🔍 DETECTOR DE FALSOS NULOS (FANTASMAS NA BASE DE DADOS)
----------------------------------------------------------------------------------------------------
✅ Excelente! Nenhum falso nulo foi detetado nas colunas de texto.
----------------------------------------------------------------------------------------------------


In [36]:
print(f"Linhas duplicadas: {df.duplicated().sum()}") #O resultado foi 96553, o que é um valor absurdo. O total de linhas do df é 8300000

#Vou calcular então a porcentagem de duplicatas, a título de curiosidade
total_duplicadas = df.duplicated().sum()
pct_duplicatas = (total_duplicadas / len(df)) * 100
print(f"Porcentagem de linhas duplicadas: {pct_duplicatas:.1f}%")

#Quero visualizar algumas dessas linhas duplicadas para entender o que está acontecendo
linhas_duplicadas = df[df.duplicated(keep=False)]
print("\n" + "-" * 100)
print("Exibindo as primeiras linhas do DataFrame com linhas duplicadas:")
print(linhas_duplicadas.head(20)) # Há realmente dados duplicados, talvez por erro na geração do arquivo ou outro.

Linhas duplicadas: 96553
Porcentagem de linhas duplicadas: 11.6%

----------------------------------------------------------------------------------------------------
Exibindo as primeiras linhas do DataFrame com linhas duplicadas:
          data  id_compra  id_cliente cl_genero  cl_estado_civil  num_filhos cl_classe  id_produto cat_produto        nome_produto
3   01/02/2019       1000         534         M                4           1         C           4   ALIMENTOS             ABACAXI
7   01/02/2019       1000         534         M                4           1         C          11   ALIMENTOS              AZEITE
14  01/02/2019       1000         534         M                4           1         C          13   ALIMENTOS              BANANA
15  01/02/2019       1000         534         M                4           1         C         218   ALIMENTOS  BIFE DE COXAO MOLE
19  01/02/2019       1000         534         M                4           1         C          13   ALIMENTOS   

# 4. Limpando e padronizando o Dataframe

In [37]:
print(f'A coluna id_compra é de tipo: {df['id_compra'].dtype}')

print(f'O total de nulos na coluna id_compra é: {df['id_compra'].isnull().sum()}')

print(f'O total de itens únicos em id_compra é: {df['id_compra'].nunique()}')

#Não parece haver problemas na coluna id_compra
#O total de linhas é 830000. Aqui o resultado foi de 18471 valores únicos, o que indica que as linhas têm valores de
#id_compra repetidos, provavelmente porque a referência é o produto comprado, e não o número da compra.

A coluna id_compra é de tipo: int64
O total de nulos na coluna id_compra é: 0
O total de itens únicos em id_compra é: 18471


In [38]:
# Já que não há nulos, vamos remover as duplicatas:
print(f"Linhas duplicadas antes da limpeza: {df.duplicated().sum()}")

df = df.drop_duplicates()
print(f"Linhas duplicadas após remoção: {df.duplicated().sum()}")

Linhas duplicadas antes da limpeza: 96553
Linhas duplicadas após remoção: 0


In [39]:
# Transformar coluna data em formato datetime

from datetime import date, time, datetime, timedelta, timezone
from zoneinfo import ZoneInfo
import calendar

# Quero verificar o formato da data (se americano ou universal)
print(f'A data máxima (em str) encontrada é: {df["data"].max()}') # O resultado foi 31/10/2021, o que indica que o formato é brasileiro (dia/mês/ano)

# Convertendo string em datetime
df['data'] = pd.to_datetime(df['data'], format='%d/%m/%Y', errors='coerce')

#Imprimindo para ter certeza de que virou datetime
print("\n" + "-" * 100)
print("DF.info para ter certeza de que virou datetime:")
print(df['data'].info(5)) # Gerou datetime, formato US

#Criando uma nova coluna, utilizando o formato DD/MM/AAAA no df
df['data_br'] = df['data'].dt.strftime('%d/%m/%Y')
print("\n" + "-" * 100)
print("DF.info com nova coluna data_br no formato DD/MM/AAAA:")
print(df.info(5)) # Gerou string na nova coluna, já que o pandas não tem um formato de data específico para o formato brasileiro.

#Reorganizando as colunas:
df = df[['data', 'data_br', 'id_compra','id_cliente','cl_genero','cl_estado_civil','num_filhos','cl_classe','id_produto','cat_produto','nome_produto']]
print("\n" + "-" * 100)
print("DF.info com colunas reorganizadas:")
print(df.info(5))

A data máxima (em str) encontrada é: 31/10/2021

----------------------------------------------------------------------------------------------------
DF.info para ter certeza de que virou datetime:
<class 'pandas.Series'>
Index: 733447 entries, 0 to 829999
Series name: data
Non-Null Count   Dtype         
--------------   -----         
733447 non-null  datetime64[us]
dtypes: datetime64[us](1)
memory usage: 11.2 MB
None

----------------------------------------------------------------------------------------------------
DF.info com nova coluna data_br no formato DD/MM/AAAA:
<class 'pandas.DataFrame'>
Index: 733447 entries, 0 to 829999
Data columns (total 11 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   data             733447 non-null  datetime64[us]
 1   id_compra        733447 non-null  int64         
 2   id_cliente       733447 non-null  int64         
 3   cl_genero        733447 non-null  str           


In [40]:
#removendo possíveis espaços antes e depois das strings com strip
for coluna in df.select_dtypes(include=['object', 'string']).columns:
    strings_com_espacos = 0
    df[coluna] = df[coluna].str.strip()
    strings_com_espacos = (df[coluna] != df[coluna].str.strip()).sum()

print("\n" + "-" * 100)
print("Quantidade de strings com espaços em branco no início ou no final por coluna:")
print(strings_com_espacos) #O resultado foi 0, então não é necessário fazer esse tipo de limpeza.


----------------------------------------------------------------------------------------------------
Quantidade de strings com espaços em branco no início ou no final por coluna:
0


In [41]:
# Verificando o que há de string fora do formato isupper
import os

if os.path.basename(os.getcwd()) == 'notebooks': os.chdir('..')
from utils.utils import verificar_maiusculas_automatico
verificar_maiusculas_automatico(df)

# Deixando tudo em maiúsculo (menos coluna data_br)
colunas_texto = df.select_dtypes(include=['object', 'string']).columns
for coluna in colunas_texto:
    df[coluna] = df[coluna].astype(str).str.upper()

# Verificando novamente o que há de string fora do formato isupper
verificar_maiusculas_automatico(df)

----------------------------------------------------------------------------------------------------
   VARREDURA DE STRINGS EM MAIÚSCULAS
❌ Coluna 'data_br': possui valores minúsculos/misturados.
   ↳ Total de termos incorretos: 733.447
✅ Coluna 'cl_genero': está 100% em maiúsculas.
✅ Coluna 'cl_classe': está 100% em maiúsculas.
✅ Coluna 'cat_produto': está 100% em maiúsculas.
❌ Coluna 'nome_produto': possui valores minúsculos/misturados.
   ↳ Total de termos incorretos: 6.451
------------------------------------------------------------
----------------------------------------------------------------------------------------------------
   VARREDURA DE STRINGS EM MAIÚSCULAS
❌ Coluna 'data_br': possui valores minúsculos/misturados.
   ↳ Total de termos incorretos: 733.447
✅ Coluna 'cl_genero': está 100% em maiúsculas.
✅ Coluna 'cl_classe': está 100% em maiúsculas.
✅ Coluna 'cat_produto': está 100% em maiúsculas.
✅ Coluna 'nome_produto': está 100% em maiúsculas.
-------------------------

{'data_br': np.int64(733447)}

In [42]:
# Substituindo os números da coluna cl_estado_civil por categorias
df['cl_estado_civil'].info() #A coluna está como inteiro, então preciso trocar para string para fazer a substituição

df['cl_estado_civil'] = df['cl_estado_civil'].astype(str)

df['cl_estado_civil'] = df['cl_estado_civil'].replace({
    '1': 'SOLTEIRO',
    '2': 'CASADO',
    '3': 'DIVORCIADO',
    '4': 'VIUVO',
    '5': 'OUTROS'
})

print(df.head(5))

<class 'pandas.Series'>
Index: 733447 entries, 0 to 829999
Series name: cl_estado_civil
Non-Null Count   Dtype
--------------   -----
733447 non-null  int64
dtypes: int64(1)
memory usage: 11.2 MB
        data     data_br  id_compra  id_cliente cl_genero cl_estado_civil  num_filhos cl_classe  id_produto cat_produto          nome_produto
0 2019-02-01  01/02/2019       1000         534         M           VIUVO           1         C          67     BEBIDAS  REFRIGERANTE GUARANA
1 2019-02-01  01/02/2019       1000         534         M           VIUVO           1         C          70     BEBIDAS   REFRIGERANTE OUTROS
2 2019-02-01  01/02/2019       1000         534         M           VIUVO           1         C         178     HIGIENE       LENCO UMEDECIDO
3 2019-02-01  01/02/2019       1000         534         M           VIUVO           1         C           4   ALIMENTOS               ABACAXI
4 2019-02-01  01/02/2019       1000         534         M           VIUVO           1         

In [43]:
#Substituindo "M" e "F" para "Masculino" e "Feminino", na coluna cl_gênero
df['cl_genero'] = df['cl_genero'].replace({
    'M': 'MASCULINO',
    'F': 'FEMININO'
})

print(df.head(5))

        data     data_br  id_compra  id_cliente  cl_genero cl_estado_civil  num_filhos cl_classe  id_produto cat_produto          nome_produto
0 2019-02-01  01/02/2019       1000         534  MASCULINO           VIUVO           1         C          67     BEBIDAS  REFRIGERANTE GUARANA
1 2019-02-01  01/02/2019       1000         534  MASCULINO           VIUVO           1         C          70     BEBIDAS   REFRIGERANTE OUTROS
2 2019-02-01  01/02/2019       1000         534  MASCULINO           VIUVO           1         C         178     HIGIENE       LENCO UMEDECIDO
3 2019-02-01  01/02/2019       1000         534  MASCULINO           VIUVO           1         C           4   ALIMENTOS               ABACAXI
4 2019-02-01  01/02/2019       1000         534  MASCULINO           VIUVO           1         C         175     LIMPEZA     LIMPADOR MULTIUSO


# 5.Fazendo uma última verificação:
### Quais os valores únicos de cada coluna? As de string podem ter variações de um mesmo termo

In [44]:
print(df.nunique())

data                 333
data_br              333
id_compra          18471
id_cliente          1000
cl_genero              2
cl_estado_civil        5
num_filhos             5
cl_classe              3
id_produto           229
cat_produto            7
nome_produto         118
dtype: int64


**Coluna nome_produto**

In [45]:
# obter valores únicos separadamente
print('-' * 100)
print("Valores únicos da coluna nome_produto:")
print(df['nome_produto'].unique()) #Não apareceu a lista completa

----------------------------------------------------------------------------------------------------
Valores únicos da coluna nome_produto:
<StringArray>
[      'REFRIGERANTE GUARANA',        'REFRIGERANTE OUTROS',
            'LENCO UMEDECIDO',                    'ABACAXI',
          'LIMPADOR MULTIUSO',           'HASTES FLEXIVEIS',
                  'MORTADELA',                     'AZEITE',
                  'AMACIANTE',                 'ENERGETICO',
 ...
                  'HAMBUGUER',      'ALIMENTO PARA PASSARO',
                     'CEBOLA',           'ENXAGUANTE BUCAL',
                      'MANGA', 'COXA E SOBRECOXA DE FRANGO',
                    'ABACATE',                      'ARROZ',
                 'ABSORVENTE',             'MOLHO BARBECUE']
Length: 118, dtype: str


In [46]:
print("\n" + "-" * 100)
print("Valores únicos da coluna nome_produto em lista:")

valores_unicos = sorted(df['nome_produto'].unique().tolist())
print(valores_unicos) #Apareceu #N/D


----------------------------------------------------------------------------------------------------
Valores únicos da coluna nome_produto em lista:
['#N/D', 'ABACATE', 'ABACAXI', 'ABSORVENTE', 'ACHOCOLATADO', 'AGUA SANITARIA', 'ALCOOL', 'ALHO', 'ALIMENTO PARA PASSARO', 'ALMONDEGA', 'AMACIANTE', 'ARROZ', 'ARROZ INTEGRAL', 'ATUM', 'AZEITE', 'AZEITONA', 'BALDE', 'BANANA', 'BATATA', 'BATATA DOCE', 'BIFE DE COXAO MOLE', 'BISCOITO', 'BROCOLIS', 'CAFE', 'CEBOLA', 'CENOURA', 'CERA', 'CHA', 'CHUPETA', 'COGUMELOS', 'CONDICIONADOR', 'COPA SUINA', 'CORACAO DE FRANGO', 'COXA E SOBRECOXA DE FRANGO', 'CREME', 'DANETTE', 'DESENGORDURANTE', 'DESINFETANTE', 'DETERGENTE', 'DOCE', 'ENERGETICO', 'ENXAGUANTE BUCAL', 'ESCOVA DE DENTE', 'FEIJAO', 'FILE DE PEIXE', 'FIO DENTAL', 'FIXADOR', 'FRALDA', 'GEL', 'GRANOLA', 'HAMBUGUER', 'HASTES FLEXIVEIS', 'HIDRATANTE', 'INSETICIDA', 'IOGURTE', 'KETCHUP', 'LATA DE ERVILHA', 'LATA DE MILHO', 'LEITE', 'LEITE CONDENSADO', 'LENCO UMEDECIDO', 'LIMAO', 'LIMPA VIDROS', 'LI

**Coluna cat_produto**

In [47]:
#Coluna cat_produto
print("\n" + "-" * 100)
print("Valores únicos da coluna nome_produto em lista:")
print(df['cat_produto'].unique().tolist()) #Apareceu #N/D


----------------------------------------------------------------------------------------------------
Valores únicos da coluna nome_produto em lista:
['BEBIDAS', 'HIGIENE', 'ALIMENTOS', 'LIMPEZA', 'ACESSORIOS', 'PET', '#N/D']


In [48]:
# Limpando os nulos encontrados apenas na última verificação

# A função que eu gerei anteriormente não previu o nulo que apareceu aqui "#N/D",
#então agora terei que tirá-lo. Depois eu poderia arrumar a função.

df['nome_produto'] = df['nome_produto'].replace('#N/D', np.nan)
df['cat_produto'] = df['cat_produto'].replace('#N/D', np.nan)

print("\n" + "-" * 100)
print("Valores únicos da coluna nome_produto em lista, após limpeza:")
print(df['nome_produto'].sort_values().unique().tolist())  # O nulo apareceu no final
print(df['cat_produto'].sort_values().unique().tolist()) # O nulo apareceu no final


----------------------------------------------------------------------------------------------------
Valores únicos da coluna nome_produto em lista, após limpeza:
['ABACATE', 'ABACAXI', 'ABSORVENTE', 'ACHOCOLATADO', 'AGUA SANITARIA', 'ALCOOL', 'ALHO', 'ALIMENTO PARA PASSARO', 'ALMONDEGA', 'AMACIANTE', 'ARROZ', 'ARROZ INTEGRAL', 'ATUM', 'AZEITE', 'AZEITONA', 'BALDE', 'BANANA', 'BATATA', 'BATATA DOCE', 'BIFE DE COXAO MOLE', 'BISCOITO', 'BROCOLIS', 'CAFE', 'CEBOLA', 'CENOURA', 'CERA', 'CHA', 'CHUPETA', 'COGUMELOS', 'CONDICIONADOR', 'COPA SUINA', 'CORACAO DE FRANGO', 'COXA E SOBRECOXA DE FRANGO', 'CREME', 'DANETTE', 'DESENGORDURANTE', 'DESINFETANTE', 'DETERGENTE', 'DOCE', 'ENERGETICO', 'ENXAGUANTE BUCAL', 'ESCOVA DE DENTE', 'FEIJAO', 'FILE DE PEIXE', 'FIO DENTAL', 'FIXADOR', 'FRALDA', 'GEL', 'GRANOLA', 'HAMBUGUER', 'HASTES FLEXIVEIS', 'HIDRATANTE', 'INSETICIDA', 'IOGURTE', 'KETCHUP', 'LATA DE ERVILHA', 'LATA DE MILHO', 'LEITE', 'LEITE CONDENSADO', 'LENCO UMEDECIDO', 'LIMAO', 'LIMPA VIDROS

# 6. Resolvendo falsos nulos '#N/D'

In [50]:
total_nd = (df == '#N/D').sum().sum()
print(f"Total de '#N/D' no DataFrame inteiro: {total_nd}")

Total de '#N/D' no DataFrame inteiro: 0


**Coluna nome_produto**

In [51]:
#Analisando a porcentagem de nulos da coluna nome_produto

print(f'O total de nulos na coluna nome_produtos é: {df["nome_produto"].isnull().sum()}')

pct_nulos_nome_produto = (df['nome_produto'].isnull().sum() / len(df)) * 100
print(f'Porcentagem de nulos na coluna nome_produto: {pct_nulos_nome_produto:.2f}%')

#A porcentagem de nulos é baixa e não é possível substituir o valr por outro, já que é um valor categórico
#Aqui, então, vou dropar as linhas
print(f'O número de linhas antes da remoção dos nulos é: {df.shape[0]}')

df = df.dropna(subset=['nome_produto'])
print(f'O número de linhas após a remoção dos nulos é: {df.shape[0]}')

print(f'O total de nulos na coluna nome_produtos depois da limpeza é: {df["nome_produto"].isnull().sum()}')

O total de nulos na coluna nome_produtos é: 3228
Porcentagem de nulos na coluna nome_produto: 0.44%
O número de linhas antes da remoção dos nulos é: 733447
O número de linhas após a remoção dos nulos é: 730219
O total de nulos na coluna nome_produtos depois da limpeza é: 0


**Coluna cat_produto**

In [52]:
#Analisando a porcentagem de nulos em cada coluna

print(f'O total de nulos no df é: {df.isnull().sum()}')

print(f'O total de nulos na coluna cat_produto é: {df["cat_produto"].isnull().sum()}')

pct_nulos_cat_produto = (df['cat_produto'].isnull().sum() / len(df)) * 100
print(f'Porcentagem de nulos na coluna cat_produto: {pct_nulos_cat_produto:.2f}%')

#A porcentagem de nulos é baixa e não é possível substituir o valr por outro, já que é um valor categórico
#Aqui, então, vou dropar as linhas
print(f'O número de linhas antes da remoção dos nulos é: {df.shape[0]}')

df = df.dropna(subset=['cat_produto'])
print(f'O número de linhas após a remoção dos nulos é: {df.shape[0]}')

print(f'O total de nulos na coluna nome_produtos é: {df["cat_produto"].isnull().sum()}')

#O resultado deu 0, o que indica que quando dropei as linhas com nulo em produto, já saíram as que tinham nulo na categoria

O total de nulos no df é: data               0
data_br            0
id_compra          0
id_cliente         0
cl_genero          0
cl_estado_civil    0
num_filhos         0
cl_classe          0
id_produto         0
cat_produto        0
nome_produto       0
dtype: int64
O total de nulos na coluna cat_produto é: 0
Porcentagem de nulos na coluna cat_produto: 0.00%
O número de linhas antes da remoção dos nulos é: 730219
O número de linhas após a remoção dos nulos é: 730219
O total de nulos na coluna nome_produtos é: 0


# 7. Exportando arquivo limpo

In [53]:
# Df final para exportação:
df_limpo = df.copy()
df_limpo = df_limpo.reset_index(drop=True)

df_limpo.head(5)

df_limpo.to_csv('data/dados_limpos.csv', sep=',', index=False, encoding='utf-8-sig')
print("Arquivo exportado com sucesso!")

#O shape do df inicialmente era Tamanho: 830000 linhas e 14 colunas
print(f'O tamanho do df_limpo é: {df_limpo.shape[0]} linhas e {df_limpo.shape[1]} colunas') # 730219 linhas e 11 colunas

Arquivo exportado com sucesso!
O tamanho do df_limpo é: 730219 linhas e 11 colunas
